# Gemma-2 9B 4-bit QLoRA training

This notebook is the individual project's full-data training workflow for
three-way LLM preference classification.

Evaluation uses the deterministic rule `id % 5 == 0`, with the evaluation
partition held out before tokenization. The full training dataset is used.

In [ ]:
# Runtime dependencies for the training workflow.
!pip install -q --no-deps transformers==4.42.3 bitsandbytes==0.43.1 accelerate==0.32.1 peft==0.11.1

In [ ]:
import ast
import json
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, log_loss

@dataclass
class Config:
    data_dir: str = os.environ.get("LLM_DATA_DIR", "/kaggle/input/llm-classification-finetuning")
    base_model: str = os.environ.get("LLM_BASE_MODEL", "unsloth/gemma-2-9b-it-bnb-4bit")
    output_dir: str = os.environ.get("LLM_OUTPUT_DIR", "gemma2_qlora_output")
    train_max_length: int = 1024
    eval_modulus: int = 5
    eval_remainder: int = 0
    epochs: int = 1
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    eval_batch_size: int = 8
    gradient_accumulation_steps: int = 2
    lora_rank: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    seed: int = 42

cfg = Config()
print(cfg)

In [ ]:
def parse_text(value):
    if isinstance(value, list):
        return " ".join(str(item) for item in value if item is not None)
    if not isinstance(value, str):
        return str(value)
    try:
        parsed = ast.literal_eval(value)
    except (SyntaxError, ValueError):
        return value
    if isinstance(parsed, list):
        return " ".join(str(item) for item in parsed if item is not None)
    return str(parsed)

train_path = Path(cfg.data_dir) / "train.csv"
frame = pd.read_csv(train_path)
required = {"id", "prompt", "response_a", "response_b", "winner_model_a", "winner_model_b", "winner_tie"}
missing = sorted(required - set(frame.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

for column in ["prompt", "response_a", "response_b"]:
    frame[column] = frame[column].map(parse_text)

eval_mask = frame["id"].astype("int64").mod(cfg.eval_modulus).eq(cfg.eval_remainder)
train_frame = frame.loc[~eval_mask].reset_index(drop=True)
eval_frame = frame.loc[eval_mask].reset_index(drop=True)

print(f"full rows: {len(frame):,}")
print(f"train rows: {len(train_frame):,}")
print(f"evaluation rows: {len(eval_frame):,}")
assert len(frame) == 57477
assert len(eval_frame) == 11476
assert len(train_frame) == 46001

In [ ]:
def class_id(a_win, b_win, tie):
    if a_win:
        return 0
    if b_win:
        return 1
    if tie:
        return 2
    raise ValueError("Each row must contain one winner label")

def format_input(prompt, response_a, response_b):
    return (
        "<prompt>: " + prompt
        + "\n\n<response_a>: " + response_a
        + "\n\n<response_b>: " + response_b
    )

def add_text_and_labels(data):
    result = data.copy()
    result["text"] = [
        format_input(p, a, b)
        for p, a, b in zip(result.prompt, result.response_a, result.response_b)
    ]
    result["labels"] = [
        class_id(a, b, t)
        for a, b, t in zip(result.winner_model_a, result.winner_model_b, result.winner_tie)
    ]
    return result[["text", "labels"]]

train_records = add_text_and_labels(train_frame)
eval_records = add_text_and_labels(eval_frame)

In [ ]:
from transformers import (
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    EvalPrediction,
    Gemma2ForSequenceClassification,
    GemmaTokenizerFast,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

tokenizer = GemmaTokenizerFast.from_pretrained(cfg.base_model)
tokenizer.add_eos_token = True
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

class PreferenceDataset(Dataset):
    def __init__(self, records):
        self.texts = records["text"].tolist()
        self.labels = records["labels"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = tokenizer(self.texts[index], max_length=cfg.train_max_length, truncation=True)
        item["labels"] = self.labels[index]
        return item

train_ds = PreferenceDataset(train_records)
eval_ds = PreferenceDataset(eval_records)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = Gemma2ForSequenceClassification.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    num_labels=3,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=["q_proj", "k_proj", "v_proj"],
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def compute_metrics(prediction: EvalPrediction):
    probabilities = torch.from_numpy(prediction.predictions).float().softmax(-1).numpy()
    labels = prediction.label_ids
    return {
        "log_loss": log_loss(labels, probabilities, labels=[0, 1, 2]),
        "accuracy": accuracy_score(labels, probabilities.argmax(axis=1)),
    }

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    overwrite_output_dir=True,
    report_to="none",
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    per_device_eval_batch_size=cfg.eval_batch_size,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    optim="adamw_8bit",
    fp16=True,
    learning_rate=cfg.learning_rate,
    seed=cfg.seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
metrics = trainer.evaluate()
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
(output_dir / "metrics.json").write_text(
    json.dumps(
        {
            "workflow": "Gemma-2 9B 4-bit QLoRA",
            "evaluation_rule": "id % 5 == 0",
            "dataset_rows": len(frame),
            "train_rows": len(train_frame),
            "evaluation_rows": len(eval_frame),
            "train_max_length": cfg.train_max_length,
            "metrics": metrics,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(metrics)

## Recorded project result

This workflow achieves evaluation log loss
`0.9371` on the deterministic `id % 5 == 0` validation split.